In [1]:
# GPU setup
import torch

# Check CUDA availability and GPU properties
print(f"torch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

torch version: 2.11.0+cu128
CUDA version: 12.8
CUDA available: True
GPU count: 1
GPU name: NVIDIA GeForce RTX 3060
VRAM: 12.9 GB


## General Purpose Statistics
Getting a general idea on the dataset before splitting it

In [6]:
import sys
import os
from pathlib import Path
sys.path.insert(0, os.path.abspath('../src/coenergy'))
from util_assessement import load_simulation_data

# Chargement du jeu de donn ées
building_simulation = "data/quentin_house"
simulations_path = Path.cwd().parent / building_simulation
simulations = load_simulation_data ( folder_path = simulations_path )

# Entr ées dynamiques (X)
temperature = simulations.indoor_temperature.round(decimals =2)
excitations = simulations.indoor_U.round(decimals =1)

# Indicateurs cibles (y)
HLC_s = simulations.HLC
Hb_labels ,Hb = simulations.Hb_labels, simulations.Hb
CDT_s = simulations.CDT
cdt_labels = ["tau_b","tau_n","tau_inf"]

# Construction du vecteur cible complet et de ses labels
indicators = torch.cat([HLC_s.unsqueeze(1),Hb,CDT_s],dim =1)
indicators_labels = ["HLC"]+list(Hb_labels) + cdt_labels


In [ ]:
assert temperature.shape[0] == excitations.shape[0] == indicators.shape[0]


In [ ]:
print (f"indicators shape: {indicators.shape} , labels: {indicators_labels}")
print (f"excitations shape: {excitations.shape} , temperature shape: {temperature.shape}")
print (f"excitations min: {excitations.min()} , max: {excitations.max()}")
print (f"excitations mean: {excitations.mean()} , std: {excitations.std()}")
print (f"temperature min: {temperature.min()} , max: {temperature.max()}")
print (f"temperature mean: {temperature.mean()} , std: {temperature.std()}")
print (f"indicators min: {indicators.min()} , max: {indicators.max()}")
print (f"indicators mean per column: {indicators.mean(dim=0)} , std per column: {indicators.std(dim=0)}")


indicators shape: torch.Size([1000000, 6]) , labels: ['HLC', 'HLC_mit', 'HLC_ext', 'tau_b', 'tau_n', 'tau_inf']
excitations shape: torch.Size([1000000, 480]) , temperature shape: torch.Size([1000000, 480])
excitations min: 0.0 , max: 7000.0
excitations mean: 3280.73779296875 , std: 3493.125244140625
temperature min: 24.219999313354492 , max: 51.54999923706055
temperature mean: 30.562271118164062 , std: 4.789175987243652
indicators min: 5.171960830688477 , max: 367.9952392578125
indicators mean per column: tensor([232.9257, 120.4881, 112.4375,  50.0013,  11.0393,  31.2310]) , std per column: tensor([31.3292, 26.2558, 17.1121, 10.0968,  2.0089,  7.5298])


In [11]:
# check if HLC = HLC_mit + HLC_ext
hlc_check = torch.allclose(indicators[:, 0], indicators[:, 1] + indicators[:, 2])
print(f"HLC check: {hlc_check}")

HLC check: True


In [22]:
# means (τ_b ≈ 50, τ_inf ≈ 31, τ_n ≈ 11) are consistent but should be checked for each simulation
# check if τ_b > τ_inf > τ_n
tau_b_check = torch.all(indicators[:, 3] > indicators[:, 5])
print(f"τ_b > τ_inf check: {tau_b_check}")

tau_inf_check = torch.all(indicators[:, 5] > indicators[:, 4])
print(f"τ_inf > τ_n check: {tau_inf_check}")

τ_b > τ_inf check: True
τ_inf > τ_n check: True


# Splitting Dataset
split the dataset into train, validation and test sets with corresponding ration of 70-15-15

In [13]:
SEED = 67
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15  

n = indicators.shape[0]

# Shuffle indices with a dedicated generator
g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(n, generator=g)

n_train = int(TRAIN_RATIO * n)
n_val = int(VAL_RATIO * n)

train_idx = perm[:n_train]
val_idx = perm[n_train:n_train + n_val]
test_idx = perm[n_train + n_val:]

def split(t: torch.Tensor):
    return t[train_idx], t[val_idx], t[test_idx]

# Inputs (X)
temp_train, temp_val, temp_test = split(temperature)
exc_train, exc_val, exc_test = split(excitations)

# Targets (y)
y_train, y_val, y_test = split(indicators)

print(f"Total: {n} | train: {len(train_idx)} | val: {len(val_idx)} | test: {len(test_idx)}")


Total: 1000000 | train: 700000 | val: 150000 | test: 150000


## Studyin the train set


In [14]:
# verify that the splits are correct
assert temp_train.shape[0] == exc_train.shape[0] == y_train.shape[0], "Train sample counts must match"


In [23]:
# Verify that the splits are correct by checking a random index
import random
index = random.randint(0, len(train_idx) - 1)
print(f"Verifying splits at index {index} of train_idx: {train_idx[index]}")

temp_matches = temp_train[index] == temperature[train_idx[index]]
exc_matches = exc_train[index] == excitations[train_idx[index]]
indicator_matches = y_train[index] == indicators[train_idx[index]]

all_temp_match = temp_matches.all().item()
all_exc_match = exc_matches.all().item()
all_indicator_match = indicator_matches.all().item()

assert all_temp_match, f"Mismatch at index {index}: temp_train[{index}] != temperature[train_idx[{index}]]"
assert all_exc_match, f"Mismatch at index {index}: exc_train[{index}] != excitations[train_idx[{index}]]"
assert all_indicator_match, f"Mismatch at index {index}: y_train[{index}] != indicators[train_idx[{index}]]"

Verifying splits at index 464549 of train_idx: 355403


In [40]:
assert not torch.isin(train_idx, val_idx).any()
assert not torch.isin(train_idx, test_idx).any()
assert not torch.isin(val_idx, test_idx).any()
assert len(torch.unique(perm)) == n
print(f"Sets don’t overlap and cover everything.")

Sets don’t overlap and cover everything.


In [16]:
# STATISTICAL ANALYSIS
# means and stds of the training set
print(f"Training indicator set means: {y_train.mean()}, stds: {y_train.std()}")
print(f"HLC mean: {y_train[:, 0].mean()}, std: {y_train[:, 0].std()}")
print(f"HLC_mit mean: {y_train[:, 1].mean()}, std: {y_train[:, 1].std()}")
print(f"HLC_ext mean: {y_train[:, 2].mean()}, std: {y_train[:, 2].std()}")

print("*"*75)
# min and max of the training set
print(f"Training indicator set min: {y_train.min()}, max: {y_train.max()}")
print(f"HLC min: {y_train[:, 0].min()}, max: {y_train[:, 0].max()}")
print(f"HLC_mit min: {y_train[:, 1].min()}, max: {y_train[:, 1].max()}")
print(f"HLC_ext min: {y_train[:, 2].min()}, max: {y_train[:, 2].max()}")
print("*"*75)
print(f"Training temperature set means: {temp_train.mean()}, stds: {temp_train.std()}")
print(f"Training excitations set means: {exc_train.mean()}, stds: {exc_train.std()}")
print(f"Training temperature set min: {temp_train.min()}, max: {temp_train.max()}")
print(f"Training excitations set min: {exc_train.min()}, max: {exc_train.max()}")

Training indicator set means: 93.01512145996094, stds: 76.60958862304688
HLC mean: 232.90884399414062, std: 31.320758819580078
HLC_mit mean: 120.46786499023438, std: 26.251873016357422
HLC_ext mean: 112.44097137451172, std: 17.11427116394043
***************************************************************************
Training indicator set min: 5.171960830688477, max: 367.9952392578125
HLC min: 111.4029312133789, max: 367.9952392578125
HLC_mit min: 42.842674255371094, max: 229.56387329101562
HLC_ext min: 56.76853561401367, max: 185.8919677734375
***************************************************************************
Training temperature set means: 30.559993743896484, stds: 4.787777900695801
Training excitations set means: 3279.115966796875, stds: 3493.023193359375
Training temperature set min: 24.219999313354492, max: 51.54999923706055
Training excitations set min: 0.0, max: 7000.0


In [24]:
# look for NaN and infinite values in the training set
nan_temp = torch.isnan(temp_train).any().item()
nan_exc = torch.isnan(exc_train).any().item()
nan_y = torch.isnan(y_train).any().item()
inf_temp = torch.isinf(temp_train).any().item()
inf_exc = torch.isinf(exc_train).any().item()
inf_y = torch.isinf(y_train).any().item()
print(f"NaN in training temperature: {nan_temp}, excitations: {nan_exc}, indicators: {nan_y}")
print(f"Infinite values in training temperature: {inf_temp}, excitations: {inf_exc}, indicators: {inf_y}")

NaN in training temperature: False, excitations: False, indicators: False
Infinite values in training temperature: False, excitations: False, indicators: False


In [17]:
print(f"Excitation count: {len(exc_train)}")

Excitation count: 700000


In [43]:
# Flag suspicious values in the training set 
temp_max = 45
hlc_max = 300
excitations_max = 6000

# create boolean masks for suspicious values
temp_mask = temp_train > temp_max
exc_mask = exc_train > excitations_max
hlc_mask = y_train[:, 0] > hlc_max

# count suspicious values
temp_suspicious_count = temp_mask.sum().item()
exc_suspicious_count = exc_mask.sum().item()
hlc_suspicious_count = hlc_mask.sum().item()

print(f"Suspicious temperature values (> {temp_max}): {temp_suspicious_count}")
print(f"Suspicious excitations values (> {excitations_max}): {exc_suspicious_count}")
print(f"Count of samples where at least one excitation is suspicious: {(exc_mask.any(dim=1)).sum().item()}")
print(f"Suspicious HLC values (> {hlc_max}): {hlc_suspicious_count}")
per_sample_max_temp = temp_train.max(dim=1).values
print("samples with max T > 45 °C:", (per_sample_max_temp > 45).sum().item())

Suspicious temperature values (> 45): 32520
Suspicious excitations values (> 6000): 157397558
Count of samples where at least one excitation is suspicious: 641010
Suspicious HLC values (> 300): 13562
samples with max T > 45 °C: 556


In [20]:
false_excitations = (~exc_mask).reshape(exc_mask.shape[0], -1).all(dim=1).sum().item()
print(f"Count of samples where all excitations are below the threshold: {false_excitations}")

Count of samples where all excitations are below the threshold: 58990
